In [1]:
#Shishir Khanal
#Sept 3, 2026
#Actor Critic Algorithm using Soft Actor Critic Algorithm
#Negative score envs do not work
#!pip install pybullet

In [2]:
import os
import torch as T
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal
import numpy as np
import gym
import pygame

# Fix compatibility between newer Gym and pybullet_envs
from collections import UserDict

registry = UserDict(gym.envs.registration.registry)
registry.env_specs = gym.envs.registration.registry
gym.envs.registration.registry = registry

import pybullet_envs
import matplotlib.pyplot as plt

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
class CriticNetwork(nn.Module):
    def __init__(self, beta, input_dims, fc1_dims, fc2_dims, n_actions, name, chkpt_dir='tmp/sac'):
        super(CriticNetwork, self).__init__()
        self.input_dims = input_dims
        self.fc1_dims = fc1_dims
        self.fc2_dims = fc2_dims
        self.n_actions = n_actions
        self.name = name
        self.checkpoint_dir = chkpt_dir
        self.checkpoint_file = os.path.join(self.checkpoint_dir, name+'_sac')

        self.fc1 = nn.Linear(self.input_dims[0] + n_actions, self.fc1_dims)
        self.fc2 = nn.Linear(self.fc1_dims, self.fc2_dims)
        self.q1 = nn.Linear(self.fc2_dims, 1)

        self.optimizer = optim.Adam(self.parameters(), lr=beta)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state, action):
        action_value = self.fc1(T.cat([state, action], dim=1))
        action_value = F.relu(action_value)
        action_value = self.fc2(action_value)
        action_value = F.relu(action_value)

        q1 = self.q1(action_value)

        return q1

    def save_checkpoint(self):
        print('Saving checkpoint...')
        T.save(self.state_dict(), self.checkpoint_file)

    def load_checkpoint(self):
        self.load_state_dict(T.load(self.checkpoint_file))

In [4]:
class ActorNetwork(nn.Module):
    def __init__(self, alpha, input_dims, fc1_dims, fc2_dims, max_action, n_actions, name, chkpt_dir='tmp/sac'):
        super(ActorNetwork, self).__init__()
        self.input_dims = input_dims
        self.fc1_dims = fc1_dims
        self.fc2_dims = fc2_dims
        self.max_action = max_action
        self.n_actions = n_actions
        self.name = name
        self.checkpoint_dir = chkpt_dir
        self.checkpoint_file = os.path.join(self.checkpoint_dir, name+'_sac')
        self.reparam_noise = 1e-6

        self.fc1 = nn.Linear(*self.input_dims, self.fc1_dims)
        self.fc2 = nn.Linear(self.fc1_dims, self.fc2_dims)
        self.mu = nn.Linear(self.fc2_dims, self.n_actions)
        self.sigma = nn.Linear(self.fc2_dims, self.n_actions)

        self.optimizer = optim.Adam(self.parameters(), lr=alpha)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state):
        prob = self.fc1(state)
        prob = F.relu(prob)
        prob = self.fc2(prob)
        prob = F.relu(prob)
        
        mu = self.mu(prob)
        sigma = self.sigma(prob)
        sigma = T.clamp(sigma, min=self.reparam_noise, max=1)
        return mu, sigma

    def sample_normal(self, state, reparametrize=True):
        mu, sigma = self.forward(state)
        probabilities = Normal(mu, sigma)

        if reparametrize:
            actions = probabilities.rsample()

        else:
            actions = probabilities.sample()
        
        action = T.tanh(actions)*T.tensor(self.max_action).to(self.device)
        log_probs = probabilities.log_prob(actions)
        log_probs-= T.log(1-action.pow(2) + self.reparam_noise)
        log_probs = log_probs.sum(1, keepdim=True)

        return action, log_probs

    def save_checkpoint(self):
        print('Saving checkpoint...')
        T.save(self.state_dict(), self.checkpoint_file)

    def load_checkpoint(self):
        print('Loading checkpoint...')
        self.load_state_dict(T.load(self.checkpoint_file))

In [5]:
class ValueNetwork(nn.Module):
    def __init__(self, beta, input_dims, fc1_dims, fc2_dims, name, chkpt_dir='tmp/sac'):
        super(ValueNetwork, self).__init__()
        self.input_dims = input_dims
        self.fc1_dims = fc1_dims
        self.fc2_dims = fc2_dims
        self.name = name
        self.checkpoint_dir = chkpt_dir
        self.checkpoint_file = os.path.join(self.checkpoint_dir, name+'_sac')

        self.fc1 = nn.Linear(*self.input_dims, self.fc1_dims)
        self.fc2 = nn.Linear(self.fc1_dims, self.fc2_dims)
        self.v = nn.Linear(self.fc2_dims, 1)

        self.optimizer = optim.Adam(self.parameters(), lr=beta)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
        self.to(self.device)

    def forward(self, state):
        state_value = self.fc1(state)
        state_value = F.relu(state_value)
        state_value = self.fc2(state_value)
        state_value = F.relu(state_value)

        v = self.v(state_value)
        return v

    def save_checkpoint(self):
        print('Saving checkpoint...')
        T.save(self.state_dict(), self.checkpoint_file)

    def load_checkpoint(self):
        print('Loading checkpoint...')
        self.load_state_dict(T.load(self.checkpoint_file))

In [6]:
#Replay buffer: Fixed size, overwrites early memories with new ones
class ReplayBuffer():
    def __init__(self, max_size, input_shape, n_actions):
        self.mem_size = max_size
        self.mem_cntr = 0
        self.state_memory = np.zeros((self.mem_size, *input_shape)) #'*' means unpack list of elements
        self.new_state_memory = np.zeros((self.mem_size, *input_shape))
        self.action_memory = np.zeros((self.mem_size, n_actions)) #For lunar lander 4 element
        self.reward_memory = np.zeros(self.mem_size)
        self.terminal_memory = np.zeros(self.mem_size, dtype=bool) #mask for setting critique values for new states to 0

    def store_transition(self, state, action, reward, state_, done):
        index = self.mem_cntr % self.mem_size
        self.state_memory[index] = state
        self.action_memory[index] = action
        self.reward_memory[index] = reward
        self.new_state_memory[index] = state_
        self.terminal_memory[index] = done
        self.mem_cntr += 1

    def sample_buffer(self, batch_size):
        max_mem = min(self.mem_cntr, self.mem_size)
        batch = np.random.choice(max_mem, batch_size)
        states = self.state_memory[batch] 
        actions = self.action_memory[batch] 
        rewards = self.reward_memory[batch]
        states_ = self.new_state_memory[batch]
        dones = self.terminal_memory[batch]

        return states, actions, rewards, states_, dones 

In [7]:
#Agent has V, Q1, Q2, Actor, target V
class Agent(nn.Module):
    def __init__(self, alpha, beta, input_dims, tau, env, 
                 env_id, layer1_size, layer2_size, gamma=0.99,
                 n_actions=2, max_size=1000000, batch_size=100, reward_scale=2):
        super(Agent, self).__init__()
        self.gamma = gamma
        self.tau = tau
        self.memory = ReplayBuffer(max_size, input_dims, n_actions)
        self.batch_size = batch_size
        self.n_actions = n_actions

        self.actor = ActorNetwork(alpha, input_dims, layer1_size, layer2_size, 
                                  n_actions=n_actions, name=env_id+'_actor', 
                                  max_action=env.action_space.high)
        self.critic_1 = CriticNetwork(beta, input_dims, layer1_size, layer2_size, n_actions=n_actions, name=env_id+'_critic_1')
        self.critic_2 = CriticNetwork(beta, input_dims, layer1_size, layer2_size, n_actions=n_actions, name=env_id+'_critic_2')
        self.value = ValueNetwork(beta, input_dims, layer1_size, layer2_size, name=env_id+'_value')
        self.target_value = ValueNetwork(beta, input_dims, layer1_size, layer2_size, name=env_id+'_target_value')

        self.scale = reward_scale
        self.update_network_parameters(tau=1)

    def choose_action(self, observation):
        state = T.Tensor([observation]).to(self.actor.device)
        actions, _ = self.actor.sample_normal(state, reparametrize=False)

        return actions.cpu().detach().numpy()[0]

    def remember(self, state, action, reward, new_state, done):
        self.memory.store_transition(state, action, reward, new_state, done)

    def update_network_parameters(self, tau=None):
        if tau == None:
            tau = self.tau

        target_value_params = self.target_value.named_parameters()
        value_params = self.value.named_parameters()

        target_value_state_dict = dict(target_value_params)
        value_state_dict = dict(value_params)

        for name in value_state_dict:
            value_state_dict[name] = tau * value_state_dict[name].clone() + (1-tau)*target_value_state_dict[name].clone()

        self.target_value.load_state_dict(value_state_dict)

    def learn(self):
        if self.memory.mem_cntr < self.batch_size:
            return
        state, action, reward, new_state, done = self.memory.sample_buffer(self.batch_size)
        reward = T.tensor(reward, dtype=T.float).to(self.critic_1.device)
        done = T.tensor(done).to(self.critic_1.device)
        state_ = T.tensor(new_state, dtype=T.float).to(self.critic_1.device)
        state = T.tensor(state, dtype=T.float).to(self.critic_1.device)
        action = T.tensor(action, dtype=T.float).to(self.critic_1.device)

        value = self.value(state).view(-1)
        value_ = self.target_value(state_).view(-1)
        value_[done] = 0.0

        actions, log_probs = self.actor.sample_normal(state, reparametrize=False)
        log_probs = log_probs.view(-1)
        q1_new_policy = self.critic_1.forward(state, actions)
        q2_new_policy = self.critic_2.forward(state, actions)
        critic_value = T.min(q1_new_policy, q2_new_policy)
        critic_value = critic_value.view(-1)

        self.value.optimizer.zero_grad()
        value_target = critic_value - log_probs
        value_loss = 0.5 * F.mse_loss(value, value_target)
        value_loss.backward(retain_graph=True)
        self.value.optimizer.step()

        actions, log_probs = self.actor.sample_normal(state, reparametrize=True)
        log_probs = log_probs.view(-1)
        q1_new_policy = self.critic_1.forward(state,actions)
        q2_new_policy = self.critic_2.forward(state, actions)
        critic_value = T.min(q1_new_policy, q2_new_policy)
        critic_value = critic_value.view(-1) #Collapse along batch dim

        actor_loss = log_probs - critic_value
        actor_loss = T.mean(actor_loss)
        self.actor.optimizer.zero_grad()
        actor_loss.backward(retain_graph=True)
        self.actor.optimizer.step()

        q_hat = self.scale * reward + self.gamma * value_
        q1_old_policy = self.critic_1.forward(state, action).view(-1) #Use action from replayed buffer
        q2_old_policy = self.critic_2.forward(state, action).view(-1)
        critic_1_loss = 0.5 * F.mse_loss(q1_old_policy, q_hat)
        critic_2_loss = 0.5 * F.mse_loss(q2_old_policy, q_hat)

        self.critic_1.optimizer.zero_grad()
        self.critic_2.optimizer.zero_grad()
        critic_loss = critic_1_loss + critic_2_loss
        critic_loss.backward()
        self.critic_1.optimizer.step()
        self.critic_2.optimizer.step()

        self.update_network_parameters()
        
    def save_models(self):
        print('Saving Models ...')
        self.actor.save_checkpoint()
        self.value.save_checkpoint()
        self.target_value.save_checkpoint()
        self.critic_1.save_checkpoint()
        self.critic_2.save_checkpoint()

    def load_models(self):
        print('Loading Models ...')
        self.actor.load_checkpoint()
        self.value.load_checkpoint()
        self.target_value.load_checkpoint()
        self.critic_1.load_checkpoint()
        self.critic_2.load_checkpoint()

In [8]:
def plot_learning_curve(scores, x, figure_file):
    running_avg = np.zeros(len(scores))
    for i in range(len(running_avg)):
        running_avg[i] = np.mean(scores[max(0, i-100):(i+1)])
    plt.plot(x, running_avg)
    plt.title('Running average of previous 100 scores')
    plt.show()
    plt.savefig(figure_file)

In [9]:
import pybullet as p

pybullet build time: Sep  3 2026 01:55:04


In [ ]:
if __name__ == '__main__':
    render_every = 49  # set to 1 to watch every episode (slow)
    env_id = 'InvertedPendulumBulletEnv-v0'
    env = gym.make(env_id).unwrapped
    env.isRender = True
    agent = Agent(alpha=0.0003, beta=0.0003, reward_scale=2, env_id=env_id,
                  input_dims=env.observation_space.shape, tau=0.005, env=env,
                  batch_size=256, layer1_size=256, layer2_size=256,
                  n_actions=env.action_space.shape[0])
    n_games = 200
    filename = env_id + '_' + str(n_games) + 'games_scale' + str(agent.scale) + '.png'
    figure_file = 'plots/' + filename

    best_score = env.reward_range[0]
    score_history = []
    load_checkpoint = False
    if load_checkpoint:
        agent.load_models()
    steps = 0

    pygame.init()
    env.reset()  # must reset before the first render() call
    frame = env.render(mode="rgb_array")  # grab one frame to get dimensions
    window = pygame.display.set_mode((frame.shape[1], frame.shape[0]))
    pygame.display.set_caption('InvertedPendulum - SAC Agent')
    episode_font = pygame.font.SysFont('Arial', 24)
    clock = pygame.time.Clock()

    for i in range(n_games):
        done = False
        observation = env.reset()
        score = 0
        render_this_episode = (i % render_every == 0)

        while not done:
            action = agent.choose_action(observation)
            observation_, reward, done, info = env.step(action)
            steps += 1
            score += reward
            if not load_checkpoint:
                agent.remember(observation, action, reward, observation_, done)
                agent.learn()
            observation = observation_  # current state = new state

            if render_this_episode:
                pygame.event.pump()  # keep OS from thinking window is frozen
                frame = env.render(mode="rgb_array")  # (H, W, 3) rgb array
                surf = pygame.surfarray.make_surface(frame.swapaxes(0, 1))
                window.blit(surf, (0, 0))

                text_surface = episode_font.render(
                    f'SAC Ep: {i + 1}/{n_games}', True, (0, 0, 0)
                )
                window.blit(text_surface, (10, 10))

                pygame.display.flip()
                clock.tick(60)

        score_history.append(score)
        avg_score = np.mean(score_history[-100:])

        if avg_score > best_score:
            best_score = avg_score
            if not load_checkpoint:
                agent.save_models()

        print('episode', i, 'score %.2f' % score,
              'trailing 100 games average score %.3f' % avg_score,
              'steps %d' % steps, env_id, 'scale', agent.scale)

    pygame.quit()
    if not load_checkpoint:
        x = [i + 1 for i in range(len(score_history))]

/tmp/ipykernel_1181360/1397788627.py:25: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  state = T.Tensor([observation]).to(self.actor.device)


Saving Models ...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
episode 0 score 14.00 trailing 100 games average score 14.000 steps 14 InvertedPendulumBulletEnv-v0 scale 2
episode 1 score 14.00 trailing 100 games average score 14.000 steps 28 InvertedPendulumBulletEnv-v0 scale 2
Saving Models ...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
episode 2 score 29.00 trailing 100 games average score 19.000 steps 57 InvertedPendulumBulletEnv-v0 scale 2
Saving Models ...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
episode 3 score 65.00 trailing 100 games average score 30.500 steps 122 InvertedPendulumBulletEnv-v0 scale 2
Saving Models ...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
Saving checkpoint...
episode 4 score 39.00 trailing 100 games average score 32.200 steps 161 Inv

In [ ]:
# if __name__ == '__main__':
#     env_id = 'InvertedPendulumBulletEnv-v0'
#     env = gym.make(env_id).unwrapped
#     env.isRender = True
#     agent = Agent(alpha=0.0003, beta=0.0003, reward_scale=2, env_id=env_id, 
#                   input_dims=env.observation_space.shape, tau=0.005, env=env, 
#                   batch_size=256, layer1_size=256, layer2_size=256, 
#                   n_actions=env.action_space.shape[0])
#     n_games = 250
#     filename = env_id + '_' + str(n_games) + 'games_scale' + str(agent.scale) + '.png'
#     figure_file = 'plots/'+filename
    
#     best_score = env.reward_range[0]
#     score_history = []
#     load_checkpoint = False
#     if load_checkpoint:
#         agent.load_models()
#     steps = 0 
#     for i in range(n_games):
#         done = False
#         observation = env.reset()
#         score = 0
#         while not done:
#             action = agent.choose_action(observation)
#             observation_, reward, done, info = env.step(action)
#             if load_checkpoint:
#                 env.render(mode='human')
#             steps += 1
#             score += reward
#             if not load_checkpoint:
#                 agent.remember(observation, action, reward, observation_, done)
#                 agent.learn()
#             observation = observation_ #current state = new state
#         score_history.append(score)
#         avg_score = np.mean(score_history[-100:])
#         if avg_score > best_score:
#             best_score = avg_score
#             if not load_checkpoint:
#                 agent.save_models()
#         print('episode', i, 'score %.2f' % score, 
#               'trailing 100 games average score %.3f' % avg_score, 
#               'steps %d' % steps, env_id, 'scale', agent.scale)
#     if not load_checkpoint:
#         x = [i+1 for i in range(len(score_history))]

In [ ]:
plot_learning_curve(score_history, x, figure_file)